In [1]:
from rdkit import Chem
from transformers import RobertaTokenizerFast, RobertaForMaskedLM, DataCollatorWithPadding


def canonicalize_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        return Chem.MolToSmiles(mol, canonical=True)
    else:
        return "Invalid SMILES"

# 示例
#miles = "CC(O)C"  # 普通 SMILES
#canonical_smiles = canonicalize_smiles(smiles)
#print("Canonical SMILES:", canonical_smiles)

In [2]:
import pickle
import pandas as pd

drug_f = pd.read_pickle("../../data/DrugComb/Process/DrugComb_all_witch_CID_data_name2_CID_dict.pkl")
print(len(drug_f))

cid_smiles = pd.read_pickle("../../data/DrugComb/Process/DrugComb_all_witch_CID_data_CID_2_SMILES_dict.pkl")
print(len(cid_smiles))

4243
4157


In [ ]:
import shap
import torch # Or tensorflow
import numpy as np
from transformers import AutoTokenizer, AutoModel # Example, adapt to your model

device = "cuda:0"
tokenizer = RobertaTokenizerFast.from_pretrained("entropy/roberta_zinc_480m", max_len=128)
model = RobertaForMaskedLM.from_pretrained('entropy/roberta_zinc_480m')
collator = DataCollatorWithPadding(tokenizer, padding=True, return_tensors='pt')

model.to(device)


 # A small representative set of SMILES
smiles_to_explain = [
    "Cc1nnc(NS(=O)(=O)c2ccc(N)cc2)s1", # Example: Ibuprofen-like structure
    "Cc1nnc(NS(=O)(=O)c2ccc(N)cc2)s1" # Example: Caffeine
]

# --- 3. Define Prediction Wrapper ---
def predict_fn(smiles_input_from_shap): # Renamed for clarity
    if isinstance(smiles_input_from_shap, np.ndarray):
        actual_smiles_list = smiles_input_from_shap.tolist()
    elif isinstance(smiles_input_from_shap, list):
        actual_smiles_list = smiles_input_from_shap
    else:
        # Attempt to convert other iterables, though SHAP usually sends ndarray or list
        try:
            actual_smiles_list = list(smiles_input_from_shap)
        except TypeError:
            print(f"Warning: predict_fn received unhandled type {type(smiles_input_from_shap)}. Returning empty output.")
            # Determine output dimension dynamically if possible, otherwise use fixed
            output_dim = 768
            if 'model' in globals() and hasattr(model, 'config') and hasattr(model.config, 'hidden_size'):
                output_dim = model.config.hidden_size
            return np.array([]).reshape(0, output_dim)
    smiles = actual_smiles_list
    inputs = collator(tokenizer(smiles)).to(device)
    outputs = model(**inputs, output_hidden_states=True)
    full_embeddings = outputs[1][-1]
    mask = inputs['attention_mask']
    embeddings = ((full_embeddings * mask.unsqueeze(-1)).sum(1) / mask.sum(-1).unsqueeze(-1))
            
    return embeddings.detach().cpu().numpy()
# Test the prediction function
print("Testing prediction function...")
sample_output = predict_fn(smiles_to_explain[:2])
print(f"Sample output shape: {sample_output.shape}") # Should be (2, 768)



In [ ]:
import pandas as pd


masker = shap.maskers.Text(tokenizer, mask_token=tokenizer.mask_token)
explainer = shap.Explainer(predict_fn, masker, algorithm="partition")

for cid in cid_smiles.keys():
    if cid in ['5311497','5394',5311497,5394]:
        sm = cid_smiles[cid]
        smiles_to_explain = [canonicalize_smiles(sm)]
        shap_values = explainer(np.array(smiles_to_explain))
        tem_df = pd.DataFrame(shap_values[0].values)
        tem_df.columns =  shap_values[0].base_values
        tem_df["smiles"] = shap_values[0].data
        tem_df.to_csv(f"../../data/SHAP/Drug/{cid}_shap.csv")

for cid in cid_smiles.keys():
    sm = cid_smiles[cid]
    smiles_to_explain = [canonicalize_smiles(sm)]
    shap_values = explainer(np.array(smiles_to_explain))
    tem_df = pd.DataFrame(shap_values[0].values)
    tem_df.columns =  shap_values[0].base_values
    tem_df["smiles"] = shap_values[0].data
    tem_df.to_csv(f"../../data/SHAP/Drug/{cid}_shap.csv")
    #break

PartitionExplainer explainer: 2it [00:56, 56.14s/it]               


  0%|          | 0/498 [00:00<?, ?it/s]

  0%|          | 0/498 [00:00<?, ?it/s]

  0%|          | 0/498 [00:00<?, ?it/s]

  0%|          | 0/498 [00:00<?, ?it/s]

...

  0%|          | 0/498 [00:00<?, ?it/s]

  0%|          | 0/498 [00:00<?, ?it/s]

  0%|          | 0/498 [00:00<?, ?it/s]

  0%|          | 0/498 [00:00<?, ?it/s]

  0%|          | 0/498 [00:00<?, ?it/s]